# Coffee17 frozen prototype geometry V1 — exploratory diagnostic
Original B0 recovery checkpoints only for feature extraction; audited recovery output and original Coffee17 required. Compare original linear classifier with train-only one- and two-center cosine decoders. No neural training, no test inference, no tuning. This is not an independent evaluation.

In [ ]:
import os, sys, subprocess, tempfile, json, zipfile, hashlib
from pathlib import Path
import torch
PIN="cdef70e2d56eabbe1c83dbdaf2e000b65375176d"
INPUT=Path("/kaggle/input")
WORK=Path("/kaggle/working")
assert INPUT.is_dir() and WORK.is_dir()
if not torch.cuda.is_available():
    raise RuntimeError("Enable Kaggle GPU")
print("TRAINING: DISABLED")
print("ANALYSIS ONLY: TRAIN/VALIDATION EMBEDDINGS; NO OOF INFERENCE")
print("CHECKPOINT DELETION: DISABLED")
print("PIN:",PIN)
REPO=Path(tempfile.mkdtemp(prefix="coffee17-prototype-audit-src-"))/"repo"
subprocess.run(["git","clone","--quiet","--no-checkout","https://github.com/ediprin/coffee-bean-classification.git",str(REPO)],check=True)
subprocess.run(["git","-C",str(REPO),"checkout","--quiet","--detach",PIN],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(REPO/"requirements/preprocessing-study.txt")],check=True)
ENV=dict(os.environ)
ENV["PYTHONPATH"]=str(REPO/"src")+os.pathsep+ENV.get("PYTHONPATH","")
sys.path.insert(0,str(REPO/"src"))
RUNNER=REPO/"src/bilinear_lmmd/experiments/run_c2_internal_oof_v1.py"
compile(RUNNER.read_text(),str(RUNNER),"exec")
from bilinear_lmmd.data.preparation.prepare_coffee17 import discover_directory_samples
from bilinear_lmmd.experiments.run_c2_internal_oof_v1 import AUTH_SHA
from bilinear_lmmd.core.reproducibility import sha256_file

# Select exactly the audited saved recovery output, not a compact analysis ZIP.
candidates=sorted(p for p in INPUT.rglob("recovery_authority.json") if p.is_file() and "datasets" not in p.relative_to(INPUT).parts)
matches=[p.parent for p in candidates if sha256_file(p)==AUTH_SHA]
if len(matches)!=1:
    print("RECOVERY AUTHORITIES FOUND:",[str(p) for p in candidates])
    raise RuntimeError("Attach exactly one saved B0/C2 recovery notebook output containing all checkpoint files.")
RECOVERY=matches[0]
print("RECOVERY SAVED OUTPUT:",RECOVERY)
# Same dataset-tree selector as recovery revision V2.
SLUG="coffee-green-bean-with-17-defects-original"
DATASETS_INPUT=INPUT/"datasets"
preferred=[DATASETS_INPUT/"sujitraarw"/SLUG,INPUT/SLUG]
present=[p for p in preferred if p.is_dir()]
if len(present)>1:
    raise RuntimeError(f"Multiple original dataset mounts: {present}")
if present:
    DATASET=present[0]
    samples=discover_directory_samples(DATASET)
else:
    valid=[]
    if DATASETS_INPUT.is_dir():
        for root in [DATASETS_INPUT]+sorted(p for p in DATASETS_INPUT.rglob("*") if p.is_dir()):
            try: samples=discover_directory_samples(root)
            except (ValueError,FileNotFoundError): continue
            if len(samples)==17 and sum(map(len,samples.values()))==979: valid.append((root,samples))
    deepest=[(root,samples) for root,samples in valid if not any(other!=root and root in other.parents for other,_ in valid)]
    if len(deepest)!=1:
        print("KAGGLE INPUT MOUNTS:",[str(p) for p in sorted(INPUT.iterdir())])
        raise RuntimeError("Attach one original Coffee17 dataset (not notebook outputs). Valid roots: "+str([str(x[0]) for x in deepest]))
    DATASET,samples=deepest[0]
assert len(samples)==17 and sum(map(len,samples.values()))==979
print("DATASET:",DATASET,"979/17")
OUT=WORK/"coffee17-prototype-geometry-v1.json"


In [ ]:
# Frozen embedding diagnostic; no optimizer, no backward, no test loader.
import copy, shutil, statistics
import numpy as np
from bilinear_lmmd.experiments.run_c2_internal_oof_v1 import audit, model_for, CLASSES, GROUPS, metric_subset
from bilinear_lmmd.data.preprocessing.study_data import build_preprocessing_evaluation_loader
from bilinear_lmmd.data.preprocessing.runtime import imagenet_normalize
from bilinear_lmmd.engine.train import classification_metrics

def unit(x):
    x=np.asarray(x,dtype=np.float64)
    n=np.linalg.norm(x,axis=-1,keepdims=True)
    if not np.isfinite(x).all() or np.any(n<=1e-12):
        raise RuntimeError("Invalid embedding norm")
    return x/n

def centers2(x):
    # Deterministic spherical two-means: initialize most central and most
    # dissimilar sample; fail closed rather than silently accepting empty clusters.
    center=unit(x.mean(axis=0,keepdims=True))[0]
    first=int(np.argmax(x@center))
    second=int(np.argmin(x@x[first]))
    if second==first: raise RuntimeError("Degenerate initial centers")
    c=x[[first,second]].copy()
    previous=None
    for iteration in range(100):
        labels=np.argmax(x@c.T,axis=1)
        counts=np.bincount(labels,minlength=2)
        if np.any(counts==0): raise RuntimeError("Empty prototype cluster")
        c=unit(np.stack([x[labels==j].mean(0) for j in range(2)]))
        if previous is not None and np.array_equal(previous,labels):
            if not np.array_equal(np.argmax(x@c.T,axis=1),labels): raise RuntimeError("Unstable assignment")
            return c,counts.tolist(),iteration+1
        previous=labels.copy()
    raise RuntimeError("Spherical two-means did not converge")

def score(y,p):
    return metric_subset(classification_metrics(y.tolist(),p.tolist(),CLASSES,GROUPS))

# Numerical smoke tests, including two separated modes.
x=unit(np.array([[1,.1],[1,-.1],[-1,.2],[-1,-.1]]))
c,counts,_=centers2(x)
assert sorted(counts)==[2,2] and c[0]@c[1]<-.99
assert np.allclose(np.linalg.norm(c,axis=1),1)
print("PROTOTYPE NUMERICAL SMOKE TEST: PASS")

if OUT.exists():
    existing=json.loads(OUT.read_text())
    if existing.get("protocol")!="coffee17-prototype-geometry-v1" or existing.get("outer_test_inference_performed") is not False:
        raise RuntimeError("Existing report differs; preserve it.")
    print("Existing complete report preserved:",OUT)
else:
    device=torch.device("cuda:0")
    torch.set_grad_enabled(False)
    # Reuse the exact previously successful integrity gate; hashes all raw bytes
    # and saved checkpoints, synthetic forwards only. Does NOT infer on test images.
    authority,folds,clean,paths,records=audit(RECOVERY,DATASET,device)
    rows=[]
    for fold in range(1,6):
        rec=records[(fold,"B0")]
        cp=rec["dir"]/"best.pt"
        assert sha256_file(cp)==rec["sha256"]
        ck=torch.load(cp,map_location="cpu",weights_only=True)
        model=model_for(rec["config"],"B0",ck).to(device).eval()
        del ck
        collected={}
        with tempfile.TemporaryDirectory(prefix=f"coffee17-prototype-fold{fold}-") as temp:
            root=Path(temp)
            cfg=copy.deepcopy(rec["config"])
            cfg["data"]["root"]=str(root)
            for split in ("train","val"):
                ids=set(folds["assignments"][f"fold_{fold}"][split])
                assert ids.isdisjoint(folds["assignments"][f"fold_{fold}"]["test"])
                for identity in sorted(ids):
                    dest=root/"source"/split/identity
                    dest.parent.mkdir(parents=True,exist_ok=True)
                    shutil.copy2(paths[identity],dest)
                    if sha256_file(dest)!=clean[identity]["sha256"]:
                        raise RuntimeError("Image digest mismatch")
                loader,classes=build_preprocessing_evaluation_loader(cfg["data"],split=split,seed=42)
                assert classes==CLASSES
                em,ys,ps=[],[],[]
                with torch.inference_mode():
                    for image,y in loader:
                        out=model(imagenet_normalize(image.to(device,non_blocking=True)))
                        em.append(out.embedding.cpu().numpy())
                        ys.append(y.numpy())
                        ps.append(out.logits.argmax(1).cpu().numpy())
                identities=[f"{Path(p).parent.name}/{Path(p).name}" for p,_ in loader.dataset.samples]
                if len(identities)!=len(ids) or set(identities)!=ids:
                    raise RuntimeError("Identity coverage mismatch")
                collected[split]=(unit(np.concatenate(em)),np.concatenate(ys),np.concatenate(ps),identities)
                del loader
        train,ytr,_,_=collected["train"]
        val,yval,linear,identities=collected["val"]
        one=[];two=[];geometry={}
        for j,name in enumerate(CLASSES):
            xj=train[ytr==j]
            mean=unit(xj.mean(0,keepdims=True))[0]
            cc,occupancy,iterations=centers2(xj)
            one.append(mean);two.append(cc)
            geometry[name]={"n_train":len(xj),"occupancy":occupancy,
                "center_cosine":float(cc[0]@cc[1]),"iterations":iterations,
                "train_mean_similarity_k1":float(np.mean(xj@mean)),
                "train_mean_similarity_k2":float(np.mean(np.max(xj@cc.T,axis=1)))}
        one=np.stack(one);two=np.stack(two)
        s1=val@one.T
        s2=(val@two.reshape(-1,train.shape[1]).T).reshape(len(val),len(CLASSES),2).max(axis=2)
        p1=s1.argmax(axis=1);p2=s2.argmax(axis=1)
        # Verify original B0 validation predictions by identity. A drift aborts
        # the diagnostic, rather than introducing a second evaluation recipe.
        import csv
        with (rec["dir"]/"validation/predictions.csv").open() as h:
            old_rows=list(csv.DictReader(h))
        old_pred={f"{Path(r['path']).parent.name}/{Path(r['path']).name}":r for r in old_rows}
        assert set(old_pred)==set(identities)
        assert all(old_pred[i]["actual"]==CLASSES[int(yval[j])] and old_pred[i]["predicted"]==CLASSES[int(linear[j])] for j,i in enumerate(identities)), "B0 validation prediction drift"
        m={"B0_linear":score(yval,linear),"K1":score(yval,p1),"K2":score(yval,p2)}
        rescue=int(np.sum((p1!=yval)&(p2==yval)));damage=int(np.sum((p1==yval)&(p2!=yval)))
        rows.append({"fold":fold,"checkpoint_sha256":rec["sha256"],"n_train":len(train),"n_val":len(val),
            "metrics":m,"geometry":geometry,"rescue_K2_vs_K1":rescue,"damage_K2_vs_K1":damage,
            "predictions":[{"identity":i,"actual":CLASSES[int(yval[j])],"B0_linear":CLASSES[int(linear[j])],"K1":CLASSES[int(p1[j])],"K2":CLASSES[int(p2[j])]} for j,i in enumerate(identities)]})
        print("FOLD",fold, {k:round(v["macro_f1"]*100,3) for k,v in m.items()},flush=True)
        del model,collected
        torch.cuda.empty_cache()
    keys=list(rows[0]["metrics"]["K1"])
    means={a:{k:statistics.mean(r["metrics"][a][k] for r in rows) for k in keys} for a in ("B0_linear","K1","K2")}
    delta={k:means["K2"][k]-means["K1"][k] for k in keys}
    positive=sum(r["metrics"]["K2"]["macro_f1"]>r["metrics"]["K1"]["macro_f1"] for r in rows)
    signal=delta["macro_f1"]>0 and positive>=3 and delta["hard_class_f1"]>=0
    report={"protocol":"coffee17-prototype-geometry-v1","scientific_pin":PIN,
      "neural_training_performed":False,"prototype_fit_train_only":True,"outer_test_inference_performed":False,
      "independent_confirmation":False,"fold_manifest_sha256":authority["fold_manifest_sha256"],
      "means":means,"K2_minus_K1":delta,"positive_macro_folds":positive,
      "decision":"EXPLORATORY_MULTICENTER_SIGNAL" if signal else "NO_CLEAR_FROZEN_MULTICENTER_SIGNAL",
      "folds":rows}
    with OUT.open("x",encoding="utf-8") as h: json.dump(report,h,indent=2,allow_nan=False)
    print("MEANS:",json.dumps(means,indent=2))
    print("DECISION:",report["decision"])
    print("UPLOAD:",OUT)
